# Group 1 Challenge: Turn-Based RPG Combat Engine

Your team is building the backend combat engine for an arcade RPG.
Fix the broken starter code across all three tasks so every assertion passes.

**Scoring:** Task 1 (`30 pts`) + Task 2 (`35 pts`) + Task 3 (`35 pts`) = **`100 pts`**.

In [ ]:
SCORES = {"task_1": 0, "task_2": 0, "task_3": 0}
print("Score tracker initialized. Complete Tasks 1 to 3 below!")

---
## Task 1: Encapsulation `[30 pts]` — Protect `Fighter` State

Fix the `Fighter` class below to enforce these rules:
1. `__init__(self, name: str, max_hp: int)`:
   - Raise `ValueError("max_hp must be positive")` if `max_hp <= 0`.
   - Store `self._name`, `self._max_hp`, initialize `self._hp = max_hp`, and `self._combat_log = []`.
2. Read-only properties:
   - `name` -> returns `self._name`
   - `max_hp` -> returns `self._max_hp`
   - `combat_log` -> returns `tuple(self._combat_log)` (so callers cannot `.append()` directly to internal history!)
3. Validated `hp` property:
   - `@property` `hp` returns `self._hp`.
   - `@hp.setter` raises `ValueError("hp must be between 0 and max_hp")` if `value < 0` or `value > self._max_hp`; otherwise sets `self._hp = value`.
4. Method `take_damage(self, amount: int) -> int`:
   - Raise `ValueError("damage cannot be negative")` if `amount < 0`.
   - Reduce `self._hp` by `amount`, clamping at `0` (`max(0, self._hp - amount)`).
   - Append `f"{self._name} took {amount} damage"` to `self._combat_log`.
   - Return updated `self._hp`.

In [ ]:
class Fighter:
    def __init__(self, name: str, max_hp: int):
        # BUG 1: Missing max_hp validation (> 0)
        # BUG 2: Attributes are public instead of protected (_name, _max_hp, _hp, _combat_log)
        self.name = name
        self.max_hp = max_hp
        self.hp = max_hp
        self.combat_log = []

    # TODO: Implement read-only properties: name, max_hp, combat_log (returning a tuple)
    # TODO: Implement hp @property and @hp.setter with range validation [0, max_hp]

    def take_damage(self, amount: int) -> int:
        # BUG 3: Does not reject negative damage, clamp at 0, or log damage
        self.hp -= amount
        return self.hp

    def attack(self, target) -> int:
        return target.take_damage(5)

    def describe(self) -> str:
        return f"{self._name} (HP: {self._hp}/{self._max_hp})"

In [ ]:
# RUN THIS CELL TO VERIFY TASK 1 (30 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    bad_init = False
    try:
        Fighter("Ghost", 0)
    except ValueError:
        bad_init = True
    assert bad_init, "Fighter('Ghost', 0) must raise ValueError when max_hp <= 0."

    f = Fighter("Kael", 100)
    assert hasattr(f, "_name") and hasattr(f, "_hp") and hasattr(f, "_combat_log"), "Use protected attributes (_name, _hp, _combat_log)."
    assert f.name == "Kael" and f.max_hp == 100 and f.hp == 100, "Properties name, max_hp, and hp must return initial values."

    read_only_blocked = False
    try:
        f.name = "Other"
    except AttributeError:
        read_only_blocked = True
    assert read_only_blocked, "Property 'name' must be read-only."

    assert isinstance(f.combat_log, tuple), "Property 'combat_log' must return a tuple so callers cannot mutate the list."

    bad_hp = False
    try:
        f.hp = 150
    except ValueError:
        bad_hp = True
    assert bad_hp, "Setting f.hp > max_hp must raise ValueError."

    bad_dmg = False
    try:
        f.take_damage(-10)
    except ValueError:
        bad_dmg = True
    assert bad_dmg, "take_damage(-10) must raise ValueError."

    remaining = f.take_damage(30)
    assert remaining == 70 and f.hp == 70, "take_damage(30) should leave hp at 70."
    assert f.take_damage(999) == 0 and f.hp == 0, "take_damage must clamp hp at 0."
    assert len(f.combat_log) == 2, "combat_log should record 2 entries."

    SCORES["task_1"] = 30
    print("[PASS] Task 1 (30 / 30 pts): Fighter encapsulation verified.")
except Exception as e:
    SCORES["task_1"] = 0
    print(f"[FAIL] Task 1 (0 / 30 pts): {e}")

---
## Task 2: Inheritance `[35 pts]` — `Warrior` and `Mage` Subclasses

Fix `Warrior` and `Mage` so they properly inherit from `Fighter`:
1. `Warrior(Fighter)`:
   - `__init__(self, name: str, max_hp: int, strength: int)`: call `super().__init__(name, max_hp)` and store `self.strength = strength`.
   - Override `attack(self, target) -> int`: deal `10 + self.strength` damage using `target.take_damage(...)` and return the damage dealt (`10 + self.strength`).
   - Extend `describe(self) -> str`: return `f"{super().describe()} | Class: Warrior (STR {self.strength})"`.
2. `Mage(Fighter)`:
   - `__init__(self, name: str, max_hp: int, spell_power: int)`: call `super().__init__(name, max_hp)` and store `self.spell_power = spell_power`.
   - Override `attack(self, target) -> int`: deal `self.spell_power * 2` damage using `target.take_damage(...)` and return the damage dealt (`self.spell_power * 2`).
   - Extend `describe(self) -> str`: return `f"{super().describe()} | Class: Mage (PWR {self.spell_power})"`.

In [ ]:
class Warrior(Fighter):
    def __init__(self, name: str, max_hp: int, strength: int):
        # BUG 1: Missing super().__init__(name, max_hp)
        self.strength = strength

    def attack(self, target) -> int:
        # BUG 2: Should deal (10 + self.strength) damage to target and return damage dealt
        return 0

    def describe(self) -> str:
        # BUG 3: Should extend super().describe()
        return f"Warrior {self.strength}"


class Mage(Fighter):
    def __init__(self, name: str, max_hp: int, spell_power: int):
        # BUG 1: Missing super().__init__(name, max_hp)
        self.spell_power = spell_power

    def attack(self, target) -> int:
        # BUG 2: Should deal (self.spell_power * 2) damage to target and return damage dealt
        return 0

    def describe(self) -> str:
        # BUG 3: Should extend super().describe()
        return f"Mage {self.spell_power}"

In [ ]:
# RUN THIS CELL TO VERIFY TASK 2 (35 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    assert issubclass(Warrior, Fighter) and issubclass(Mage, Fighter), "Warrior and Mage must inherit from Fighter."

    dummy = Fighter("TargetDummy", 200)
    w = Warrior("Arthur", 120, strength=8)
    m = Mage("Lyra", 80, spell_power=12)

    assert w.name == "Arthur" and w.hp == 120, "Warrior must call super().__init__() to set Fighter state."
    assert m.name == "Lyra" and m.hp == 80, "Mage must call super().__init__() to set Fighter state."

    d1 = w.attack(dummy)
    assert d1 == 18 and dummy.hp == 182, f"Warrior.attack should deal 18 damage (got {d1}, dummy hp {dummy.hp})."

    d2 = m.attack(dummy)
    assert d2 == 24 and dummy.hp == 158, f"Mage.attack should deal 24 damage (got {d2}, dummy hp {dummy.hp})."

    assert w.describe() == "Arthur (HP: 120/120) | Class: Warrior (STR 8)", f"Unexpected Warrior.describe(): {w.describe()}"
    assert m.describe() == "Lyra (HP: 80/80) | Class: Mage (PWR 12)", f"Unexpected Mage.describe(): {m.describe()}"

    SCORES["task_2"] = 35
    print("[PASS] Task 2 (35 / 35 pts): Inheritance and super() verified.")
except Exception as e:
    SCORES["task_2"] = 0
    print(f"[FAIL] Task 2 (0 / 35 pts): {e}")

---
## Task 3: Polymorphism and Duck Typing `[35 pts]` — `TrapTurret` & `run_battle_round`

Fix the code below to achieve clean polymorphism:
1. `TrapTurret` must **NOT** inherit from `Fighter` (keep it standalone to demonstrate duck typing).
   - Rename/implement its method `attack(self, target) -> int` so it deals `self.damage` via `target.take_damage(self.damage)` and returns `self.damage`.
2. Refactor `run_battle_round(units: list, target: Fighter) -> int`:
   - Remove **all** `isinstance(...)` checks!
   - Loop through every unit in `units`, call `unit.attack(target)`, and return the sum of damage dealt.

In [ ]:
class TrapTurret:
    def __init__(self, label: str, damage: int):
        self.label = label
        self.damage = damage

    # BUG 1: Wrong method name for duck typing! Should be attack(self, target) -> int
    def fire_bolt(self, target) -> int:
        target.take_damage(self.damage)
        return self.damage


def run_battle_round(units: list, target: Fighter) -> int:
    # BUG 2: Fragile isinstance() branching breaks on duck-typed TrapTurret!
    # Refactor into a clean polymorphic loop with ZERO isinstance() calls.
    total_damage = 0
    for u in units:
        if isinstance(u, Warrior):
            total_damage += u.attack(target)
        elif isinstance(u, Mage):
            total_damage += u.attack(target)
    return total_damage

In [ ]:
# RUN THIS CELL TO VERIFY TASK 3 (35 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    assert not issubclass(TrapTurret, Fighter), "TrapTurret must NOT inherit from Fighter (duck typing test)."
    assert hasattr(TrapTurret, "attack"), "TrapTurret must implement an attack(self, target) method."

    assert "isinstance" not in run_battle_round.__code__.co_names, "Remove all isinstance() calls from run_battle_round()."

    boss = Fighter("Golem", 200)
    team = [Warrior("W1", 100, 5), Mage("M1", 80, 10), TrapTurret("Laser", 12)]
    total = run_battle_round(team, boss)
    # 15 + 20 + 12 = 47
    assert total == 47 and boss.hp == 153, f"Expected total damage 47 and boss hp 153 (got {total}, hp {boss.hp})."

    SCORES["task_3"] = 35
    print("[PASS] Task 3 (35 / 35 pts): Polymorphism and duck typing verified.")
except Exception as e:
    SCORES["task_3"] = 0
    print(f"[FAIL] Task 3 (0 / 35 pts): {e}")

---
## Final Scoreboard

In [ ]:
# RUN THIS CELL TO VIEW YOUR FINAL COMPETITION SCORE
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
total = SCORES["task_1"] + SCORES["task_2"] + SCORES["task_3"]
print("=" * 42)
print("DS212 GROUP COMPETITION SCOREBOARD")
print("=" * 42)
print(f"Task 1 (Encapsulation): {SCORES['task_1']:>2} / 30 pts")
print(f"Task 2 (Inheritance):   {SCORES['task_2']:>2} / 35 pts")
print(f"Task 3 (Polymorphism):  {SCORES['task_3']:>2} / 35 pts")
print("-" * 42)
print(f"TOTAL SCORE:           {total:>3} / 100 pts")
if total == 100:
    print("STATUS: ALL CHALLENGES COMPLETED [100%]")
else:
    print("STATUS: IN PROGRESS — KEEP DEBUGGING!")
print("=" * 42)